In [8]:
from delta.tables import DeltaTable
from pyspark.sql.functions import md5,col,concat_ws,date_format,year, dayofmonth,month,to_date


StatementMeta(, 2bc9bdf0-1349-46d1-824c-3f2aca0b9a31, 10, Finished, Available, Finished, False)

In [9]:
def merge_tables(dataframe, table_name,condition):
   if spark.catalog.tableExists(table_name):
       table_old= DeltaTable.forName(spark,table_name)
       table_old.alias("old").merge(
           dataframe.alias("new"),
           condition
       ).whenNotMatchedInsertAll().execute()
       print("the table is incrementally loaded")
   
   else:
       dataframe.write.format("delta").saveAsTable(table_name)
       print("the table is full loaded")
   


StatementMeta(, 2bc9bdf0-1349-46d1-824c-3f2aca0b9a31, 11, Finished, Available, Finished, False)

In [10]:
jobs = spark.read.format("delta").table("silver_jobs")

StatementMeta(, 2bc9bdf0-1349-46d1-824c-3f2aca0b9a31, 12, Finished, Available, Finished, False)

In [11]:
display(jobs.limit(2))

StatementMeta(, 2bc9bdf0-1349-46d1-824c-3f2aca0b9a31, 13, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e95f7b49-0b64-4c3e-8b23-0ae9229fbfa5)

## **Dim Companies**

In [16]:
company_dim = jobs.withColumn("company_key",md5(col("company_name")))\
.select(
    col("company_key"),
    col("company_name")
).distinct()


display(company_dim.limit(3))

StatementMeta(, 2bc9bdf0-1349-46d1-824c-3f2aca0b9a31, 18, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b3ba6b1f-121c-43d0-9d4b-93e0154278e1)

In [17]:
merge_tables(company_dim,"dim_company","new.company_key = old.company_key")

StatementMeta(, 2bc9bdf0-1349-46d1-824c-3f2aca0b9a31, 19, Finished, Available, Finished, False)

the table is incrementally loaded


## **Dim Jobs**

In [18]:
dim_jobs = jobs.select("job_title","category","job_type").withColumn("job_id",md5(concat_ws("||",col("job_title"),col("category")))).distinct()

StatementMeta(, 2bc9bdf0-1349-46d1-824c-3f2aca0b9a31, 20, Finished, Available, Finished, False)

In [19]:
merge_tables(dim_jobs,"dim_jobs","new.job_id = old.job_id")

StatementMeta(, 2bc9bdf0-1349-46d1-824c-3f2aca0b9a31, 21, Finished, Available, Finished, False)

the table is full loaded


In [20]:
display(dim_jobs.limit(3))

StatementMeta(, 2bc9bdf0-1349-46d1-824c-3f2aca0b9a31, 22, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b5e2525d-8bad-4c6f-a405-ab0550abc547)

## **Dim Location**

In [22]:
dim_location = jobs.select("location").distinct()\
.withColumn("location_id",md5(col("location")))

StatementMeta(, 2bc9bdf0-1349-46d1-824c-3f2aca0b9a31, 24, Finished, Available, Finished, False)

In [23]:
merge_tables(dim_location,"dim_location","old.location_id= new.location")

StatementMeta(, 2bc9bdf0-1349-46d1-824c-3f2aca0b9a31, 25, Finished, Available, Finished, False)

the table is full loaded


In [24]:
display(dim_location.limit(3))

StatementMeta(, 2bc9bdf0-1349-46d1-824c-3f2aca0b9a31, 26, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a491c3dc-84e7-4284-8d6f-461739984cf2)

## **Dim Date**

In [25]:
dim_date = jobs.select("job_date").distinct()\
.withColumn("date_id",to_date(col("job_date")))\
.withColumn("year",year(col("job_date")))\
.withColumn("month",month(col("job_date")))\
.withColumn("day_of_month",dayofmonth(col("job_date")))\
.withColumn("month_name",date_format(col("job_date"),"MMMM"))\
.withColumn("day_name",date_format(col("job_date"),"EEEE")).select("date_id","year","month","day_of_month","day_name")

StatementMeta(, 2bc9bdf0-1349-46d1-824c-3f2aca0b9a31, 27, Finished, Available, Finished, False)

In [26]:
merge_tables(dim_date,"dim_date","old.date_id = new.date_id")

StatementMeta(, 2bc9bdf0-1349-46d1-824c-3f2aca0b9a31, 28, Finished, Available, Finished, False)

the table is full loaded


## **Fact Df**

In [27]:
fact_df = jobs.alias("f").withColumn("company_id",md5(col("company_name")))\
.withColumn("job_id_dim",md5(concat_ws("||",col("job_title"),col("category"))))\
.withColumn("location_id",md5(col("location")))\
.withColumn("date_id",to_date(col("job_date")))\
.select(
    col("f.job_id").alias("fact_id"),
    col("date_id"),
    col("company_id"),
    col("job_id_dim").alias("job_id"),
    col("location_id"),
    col("f.min_salary").cast("int"),
    col("f.max_salary").cast("int"),
    col("f.avg_salary").cast("double")
)

StatementMeta(, 2bc9bdf0-1349-46d1-824c-3f2aca0b9a31, 29, Finished, Available, Finished, False)

In [28]:
merge_tables(fact_df,"fact_table","old.fact_id = new.fact_id")

StatementMeta(, 2bc9bdf0-1349-46d1-824c-3f2aca0b9a31, 30, Finished, Available, Finished, False)

the table is full loaded


## **Dim Location**